# Corpus Intelligence -- Deep Analytics Across GLP-1 Literature

**The value isn't in one document -- it emerges from connecting hundreds.** This notebook turns a collection of open-access clinical papers on **GLP-1 receptor agonists** (semaglutide, tirzepatide, liraglutide, dulaglutide, orforglipron) into two layers of intelligence:

1. **Per-paper** -- a significance assessment plus structured trial fields (drug, phase, sample size, endpoint, hazard ratio, p-value, sponsor, NCT id), with quantitative results read off the **figures** as well as the text.
2. **Cross-document** -- trend rollups by drug/phase/year and an **AI-written competitive-landscape briefing per drug** that synthesizes what all of a drug's studies collectively show.

Every step is a **declarative, incremental Snowflake Cortex pipeline**: each AI function runs once per *new* paper, and the same SQL that runs on ~50 papers runs on millions.

```
DEMO_RES_DOCS_STAGE (paper PDFs + per-page PNGs)  ->  stream + task  ->  DEMO_RES_FILE_LOG
  -> DT_DEMO_RES_PARSED     AI_PARSE_DOCUMENT(LAYOUT)        full paper text
  -> DT_DEMO_RES_FIGURES    AI_COMPLETE(vision) per page     figure findings (or NO_FIGURE)
  -> DT_DEMO_RES_FIG_AGG    GROUP BY paper                   one row / paper
  -> DT_DEMO_RES_ENRICHED   parsed text (+) figure findings  (figure-only numbers become extractable)
  -> DT_DEMO_RES_ASSESSED   AI_COMPLETE(json)                summary / primary finding / significance
  -> DT_DEMO_RES_ENTITIES   AI_EXTRACT                       drug / phase / n / endpoint / HR / p / sponsor / NCT
  -> DT_DEMO_RES_PAPER      assemble per paper               -> DEMO_RES_PAPER_INTELLIGENCE (view)
  -> DT_DEMO_RES_TRENDS     GROUP BY drug/phase/year         -> DEMO_RES_TRENDS (view)
  -> DT_DEMO_RES_LANDSCAPE  GROUP BY drug -> AI_COMPLETE      -> DEMO_RES_LANDSCAPE (view)  [the synthesis]
```

All nine dynamic tables refresh `INCREMENTAL`. Intermediate tables carry `TARGET_LAG = DOWNSTREAM`; the two cross-document terminals take the user lag (1 hour).

> **Before running:** this notebook reads objects the pipeline already built, so first run `00_setup.sql`, the sourcing script (`source_corpus_intelligence.py`, which also backfills the file log and loads the paper dimension), `10_pipeline.sql` (creates the dynamic tables, zero-spend), and `20_analytics.sql` section A (the cost-gated AI refresh) -- then let the dynamic tables settle. Substitute `{database}` / `{schema}` / `{warehouse}` in the context cell below; all other object references resolve against the schema it sets.

In [ ]:
USE SCHEMA {database}.{schema};
USE WAREHOUSE {warehouse};

## 1 - The corpus

Open-access papers sourced from **Europe PMC** (which also indexes medRxiv / bioRxiv preprints), in per-drug, trial-biased buckets so the per-drug synthesis has depth.

In [ ]:
SELECT DRUG_DISPLAY,
       COUNT(*)   AS papers,
       MIN(YEAR)  AS earliest,
       MAX(YEAR)  AS latest
FROM DEMO_RES_PAPER_INTELLIGENCE
GROUP BY DRUG_DISPLAY
ORDER BY papers DESC;

The pipeline is built from incremental dynamic tables -- every one refreshes only on *new* files.

In [ ]:
SHOW DYNAMIC TABLES LIKE 'DT_DEMO_RES%';
SELECT "name", "refresh_mode", "target_lag", "scheduling_state"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY "name";

## 2 - Per-paper intelligence

`AI_PARSE_DOCUMENT` turns each PDF into text; `AI_COMPLETE` (vision) reads the figures; an `AI_COMPLETE` JSON judgment assigns **significance**; and `AI_EXTRACT` pulls the structured trial fields. Note the quantitative columns (hazard ratio, p-value) -- many are read off **figures**.

In [ ]:
SELECT DRUG_DISPLAY, YEAR, TRIAL_PHASE, SAMPLE_SIZE, SIGNIFICANCE,
       PRIMARY_FINDING, HAZARD_RATIO, P_VALUE, NCT_ID
FROM DEMO_RES_PAPER_INTELLIGENCE
ORDER BY DRUG_DISPLAY, YEAR DESC
LIMIT 20;

How the significance judgment distributes across the corpus:

In [ ]:
SELECT SIGNIFICANCE, COUNT(*) AS papers
FROM DEMO_RES_PAPER_INTELLIGENCE
GROUP BY SIGNIFICANCE
ORDER BY papers DESC;

Figure coverage -- how many staged pages actually carried a data figure, per drug:

In [ ]:
SELECT rp.DRUG,
       COUNT(*)                                       AS pages,
       COUNT_IF(f.FIG_FINDING NOT ILIKE 'NO_FIGURE%') AS pages_with_figures
FROM DT_DEMO_RES_FIGURES f
JOIN DEMO_RES_PAPERS rp ON rp.PAPER_ID = f.PAPER_ID
GROUP BY rp.DRUG
ORDER BY rp.DRUG;

## 3 - Cross-document trends

Plain incremental aggregates over the per-paper table -- studies, mean enrollment, and high-significance counts by drug and phase.

In [ ]:
SELECT DRUG_DISPLAY, TRIAL_PHASE,
       SUM(N_STUDIES)              AS studies,
       ROUND(AVG(AVG_SAMPLE_SIZE)) AS avg_enrollment,
       SUM(N_HIGH_SIGNIFICANCE)    AS high_significance_studies
FROM DEMO_RES_TRENDS
GROUP BY DRUG_DISPLAY, TRIAL_PHASE
ORDER BY DRUG_DISPLAY, TRIAL_PHASE;

## 4 - Competitive landscape -- the synthesis

The payoff: for each drug, `AI_COMPLETE` reads every per-paper finding and writes a comparative briefing -- efficacy, strength/consistency of evidence, and positioning within the GLP-1 class. This is a dynamic table: as new papers land, the affected drug's briefing re-synthesizes.

In [ ]:
SELECT DRUG_DISPLAY, N_STUDIES, BRIEFING
FROM DEMO_RES_LANDSCAPE
ORDER BY N_STUDIES DESC;

## 5 - Multi-hop question answering

Ask a question that requires reasoning *across* the corpus. The cell below assembles the per-paper findings into context and runs one live `AI_COMPLETE` call (this cell spends a small amount).

In [ ]:
WITH ctx AS (
  SELECT LISTAGG(
           DRUG_DISPLAY || ' | ' || COALESCE(TRIAL_PHASE,'?') || ' | n='
           || COALESCE(SAMPLE_SIZE::STRING,'?') || ' | '
           || COALESCE(PRIMARY_FINDING, OUTCOME, PRIMARY_ENDPOINT_RESULT, 'n/a'),
           '\n'
         ) WITHIN GROUP (ORDER BY DRUG_DISPLAY) AS BODY
  FROM DEMO_RES_PAPER_INTELLIGENCE
)
SELECT AI_COMPLETE('claude-4-sonnet',
  'Using ONLY this table of GLP-1 trial findings, answer concisely: which drug shows the largest '
  || 'weight-loss effect, with what number, and in how many studies is weight loss reported? '
  || 'Name the drugs compared.\n\nFindings:\n' || BODY) AS ANSWER
FROM ctx;

## Scale

This pipeline processed tens of papers, but nothing about it is sized for tens. Every dynamic table is `INCREMENTAL`: an AI function runs **once per new paper**, and the trend tables and landscape briefings refresh within their `TARGET_LAG` when (and only when) new data lands. The same declarative SQL runs unchanged on millions of documents -- **cost scales with new files, not corpus size.** Drop more PDFs on the stage and the per-drug briefings update themselves.

> **Text-only variant:** drop `DT_DEMO_RES_FIGURES` / `DT_DEMO_RES_FIG_AGG` and read the enriched text straight from the parsed body -- the significance judgment, extraction, trends, and landscape synthesis are unchanged; only figure-only numbers are lost. (`source_corpus_intelligence.py --skip-render` builds the matching corpus.)